# ⚡ Retune Siêu Tốc (Nhánh Test)
**Mục tiêu:** Tinh chỉnh Threshold nhanh gọn từ các file kết quả cũ mà không cần chạy lại Retrieval.

### 📁 File cần có trong Drive > `R2AI_Artifacts`:
- `retrieval.db` (Database chứa điểm RRF và Candidates) ✅
- `results_partial.jsonl` (File backup generation nếu có) ✅
- `R2AIStage1DATA.json` (Đề bài) ✅

## Cell 1 — Mount Drive & Chuẩn bị

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'\n📋 Scan {DRIVE_DIR}:')
for root, dirs, files in os.walk(DRIVE_DIR):
    level = root.replace(DRIVE_DIR, '').count(os.sep)
    indent = '  ' * level
    rel = root.replace(DRIVE_DIR, '') or '/'
    print(f'{indent}📁 {rel}/')
    for f in files:
        sz = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
        print(f'{indent}  📄 {f} ({sz:.1f}MB)')
print('\n✅ Cell 1 Done!')


## Cell 2 — Git Clone & Restore Artifacts

In [ ]:
import os, sys, shutil, subprocess
DRIVE_DIR = '/content/drive/MyDrive/R2AI_Artifacts'
WORK_DIR  = '/content/sme-legal-assistant'
REPO_URL  = 'https://github.com/Platypus27-coder/sme-legal-assistant.git'
BRANCH    = 'test'

try:
    os.chdir('/content')
except Exception: pass
try:
    os.chdir('/content')
except Exception: pass
try:
    os.chdir('/content')
except Exception: pass
try:
    os.chdir('/content')
except Exception: pass
try:
    os.chdir('/content')
except Exception: pass
try:
    os.chdir('/content')
except Exception: pass
try:
    os.chdir('/content')
except Exception: pass
if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)

print(f'⬇️ Clone branch {BRANCH} từ Github...')
r = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, WORK_DIR], capture_output=True, text=True)
if r.returncode != 0:
    print(f'❌ Lỗi Clone: {r.stderr}')
    raise SystemExit

# Tạo thư mục
for d in ['output', 'cache', 'raw', 'index']:
    os.makedirs(f'{WORK_DIR}/artifacts/{d}', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)

# Copy files từ Drive vào Project
print('📦 Restore Artifacts từ Drive...')
files_to_restore = [
    (['retrieval_hybrid.db', 'retrieval.db'], 'artifacts/cache/retrieval.db'),
    ('results_partial.jsonl', 'artifacts/output/results_partial.jsonl'),
    ('R2AIStage1DATA.json', 'data/R2AIStage1DATA.json')
]

for src_names, dst_path in files_to_restore:
    if isinstance(src_names, str):
        src_names = [src_names]
    restored = False
    for src_name in src_names:
        src_full = f'{DRIVE_DIR}/{src_name}'
        if os.path.exists(src_full):
            shutil.copy2(src_full, f'{WORK_DIR}/{dst_path}')
            print(f'  ✅ Restored: {src_name}')
            restored = True
            break
    if not restored:
        print(f'  ⚠️ Missing: {src_names[0]} (Thực thi có thể lỗi nếu thiếu file này)')

sys.path.insert(0, f'{WORK_DIR}/src')
os.chdir(WORK_DIR)
print('\n✅ Cell 2 Done!')

## Cell 3 — Cài đặt thư viện nhẹ

In [ ]:
import subprocess, sys
print('📦 Cài đặt thư viện...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers>=2.7.0'], capture_output=True, text=True)
print('✅ Cell 3 Done!')


## Cell 4 — Tinh chỉnh Threshold (Retune)

In [ ]:
import subprocess

# ── Chỉnh sửa các tham số ở đây để tối ưu hóa F2-Macro ──
HIGH_CONF   = 0.62   # Điểm tối thiểu để chọn làm Điều luật liên quan
SAFE        = 0.52   # Điểm tối thiểu để làm context LLM (fallback)
MAX_ART     = 3      # Tối đa số Điều luật nộp mỗi câu (đặt 4 hoặc 5 để tăng Recall cho F2)
MIN_ART     = 0      # Ít nhất phải nộp 1 Điều luật (không nộp mảng rỗng)
NO_RERANKER = False  # Đặt là False để BẬT Reranker (Cực kỳ quan trọng để đạt điểm tối ưu với E5)
# ────────────────────────────────────────────────────────

print('🚀 Bắt đầu chấm điểm lại (Retune)...')
cmd = [
    'python', 'rerank_retune.py',
    '--high-conf', str(HIGH_CONF),
    '--safe',      str(SAFE),
    '--max-art',   str(MAX_ART),
    '--min-art',   str(MIN_ART),
]
if NO_RERANKER:
    cmd.append('--no-reranker')

r = subprocess.run(cmd, capture_output=False)

print('\n🎉 HOÀN TẤT! File submission_reranked.zip đã được lưu vào Google Drive của bạn.')

## Cell 5 — (Bứt phá) Tạo tự động cả 5 Variant tối ưu phiên bản V5

Chúng ta đã xác định được đỉnh của ngưỡng lọc nằm chính xác ở khoảng **0.67 - 0.69** (ngưỡng 0.68 đạt 0.4483, nâng lên 0.70 điểm bị tụt xuống 0.4383). Phiên bản V5 tập trung khai thác đỉnh này bằng cách mở rộng sang `max_articles=3` nhưng giữ ngưỡng siêu tin cậy để vừa ăn trọn các câu 3 luật, vừa giữ sạch các câu 1-2 luật:
- `submission_pure_max3_t68_v5.zip` (Ngưỡng 0.68, tối đa 3 bài - Dự đoán: **0.455 - 0.465** - Ứng viên số 1!)
- `submission_pure_max3_t67_v5.zip` (Ngưỡng 0.67, tối đa 3 bài)
- `submission_pure_max3_t69_v5.zip` (Ngưỡng 0.69, tối đa 3 bài)
- `submission_pure_max2_t67_v5.zip` (Ngưỡng 0.67, tối đa 2 bài)
- `submission_pure_max2_t69_v5.zip` (Ngưỡng 0.69, tối đa 2 bài)

In [ ]:
import subprocess
import torch

has_gpu = torch.cuda.is_available()
print('🔍 Kiểm tra thiết bị phần cứng:')
print(f'   - CUDA Available: {has_gpu}')
if has_gpu:
    print(f'   - GPU Device: {torch.cuda.get_device_name(0)}')
else:
    print('   ⚠️ WARNING: Không tìm thấy GPU! Việc chạy Reranker trên CPU sẽ cực kỳ lâu (mất khoảng 2 tiếng).')
    print('   💡 Khuyên nghị: Hãy bấm Dừng cell này, vào "Chỉnh sửa" -> "Cài đặt sổ ghi chép" -> Chọn GPU T4 rồi chạy lại.')

print('\n🚀 Bắt đầu tạo 5 Variant tối ưu nộp bài (tiến độ sẽ được hiển thị trực tiếp)...')
cmd = [
    'python', '-u', 'tools/optimize_submission.py',
    '--use-reranker',
    '--device', 'cuda' if has_gpu else 'cpu',
    '--batch-size', '64'
]

try:
    # Chạy python với cờ -u (unbuffered) để hiển thị log real-time
    subprocess.run(cmd, check=True)
    print('\n🎉 HOÀN TẤT! Cả 5 file submission đã có trên Drive R2AI_Artifacts của bạn.')
except subprocess.CalledProcessError as e:
    print(f'\n❌ Lỗi thực thi optimize_submission.py: {e}')
